# 第 2 章 §2.4 · Google ADK（Auto-Flow 隐式路由）

代码里**没有任何路由判断**：没有 if 分支、没有标签比较。你和模型之间的契约只有三样——
协调者的 `instruction`、每个子 agent 的 `description`、`sub_agents=[...]`。

对比 §2.3：那里你当调度员（1 次模型调用）；这里你写招聘启事让模型自己选人（≥2 次调用）。

kernel 必须选 **`Python (agentic-adk)`**。原代码在 ADK 2.13 上跑不起来，三处【适配】已标。
Auto-Flow 内部时序、四个实测发现、和 §2.3 的选型对比都在 `../../ch02_routing/README.md`。

In [1]:
import os
import re
import time
import uuid

import sys
from pathlib import Path

# 【适配】向上找含 common/ 的目录（后端在官方 / 中转站之间切换，见 common/adk.py）
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, adk_backend, describe_adk

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types

# 【适配 1】书里 gemini-2.0-flash 已下线。后端（官方 / 中转站）由 adk_model() 决定，
#   切换只改 .env 的 ADK_BACKEND，这里不动。
MODEL = adk_model()
print("使用模型:", describe_adk())


# docstring 就是工具的 description —— 而 description 正是模型判断「该不该调这个工具」的依据。
# 也就是说：docstring 也是路由逻辑。类型注解会被转成 JSON Schema，模型靠它决定传什么参数。
def booking_handler(request: str) -> str:
    """Handles booking requests for flights and hotels.

    Args:
        request: The user's request for a booking.

    Returns:
        A confirmation message that the booking was handled.
    """
    print("  >>> Booking Handler Called")
    return f"Booking action for '{request}' has been simulated."


def info_handler(request: str) -> str:
    """Handles general information requests.

    Args:
        request: The user's question.

    Returns:
        A message indicating the information request was handled.
    """
    print("  >>> Info Handler Called")
    return f"Information request for '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    print("  >>> Unclear Handler Called")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

# ⚠️ unclear_handler 在整段代码里从未被注册成工具、也没挂到任何 agent 上（书里第 11 页自己承认）。
#    §2.3 里它至少还是 RunnableBranch 的默认分支，这里连"兜底"这个概念都消失了。

booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)

# ★ 本章新增：Auto-Flow 靠 description 挑人 —— 写错 description 就路由错，
#   而且和 §2.3 改 prompt 一样：没有 diff、没有测试会失败。
booking_agent = Agent(
    name="Booker",
    model=MODEL,
    description="A specialized agent that handles all flight and hotel booking requests by calling the booking tool.",
    tools=[booking_tool],
)

info_agent = Agent(
    name="Info",
    model=MODEL,
    description="A specialized agent that provides general information and answers user questions by calling the info tool.",
    tools=[info_tool],
)

coordinator = Agent(
    name="Coordinator",
    model=MODEL,
    instruction=(
        "You are the main coordinator. Your only task is to analyze incoming user requests "
        "and delegate them to the appropriate specialist agent. Do not try to answer the user directly.\n"
        "- For any requests related to booking flights or hotels, delegate to the 'Booker' agent.\n"
        "- For all other general information questions, delegate to the 'Info' agent."
    ),
    description="A coordinator that routes user requests to the correct specialist agent.",
    # ★ 这一行就是开关：只要存在 sub_agents，ADK 的 Auto-Flow 自动启用，
    #   模型获得「把对话转交给某个子 agent」的能力。删掉它，coordinator 退化成普通 agent。
    sub_agents=[booking_agent, info_agent],
)


def extract_text(content) -> str:
    """从 event.content 里取文本。

    【适配】本机实测 Content 对象没有 .text 属性，必须走 parts 分支，否则 AttributeError。
    """
    if getattr(content, "text", None):
        return content.text
    parts = getattr(content, "parts", None)
    if parts:
        return "".join(p.text for p in parts if getattr(p, "text", None))
    return ""


def _retry_delay(msg: str, default: int = 30) -> int:
    """从 429 错误体里抠出服务端建议的等待秒数（形如 retryDelay: '18s'）。"""
    m = re.search(r"retryDelay['\"]?\s*:\s*['\"]?(\d+(?:\.\d+)?)s", msg)
    if m:
        return min(int(float(m.group(1))) + 5, 120)   # 留 5s 余量
    return default


def run_coordinator(runner: InMemoryRunner, request: str, *,
                    max_retry: int = 2, verbose: bool = False):
    """跑一个请求，从事件流里取出最终回复。

    【适配 2】create_session 是协程，同步上下文要用 create_session_sync，
              否则 runner.run 报 Session not found。
    【适配 3】免费层 5 RPM：重试时用全新 session（上次失败已把用户消息写进去了），
              并优先读服务端返回的 retryDelay，而不是固定等待。
    """
    print(f"\n--- Running Coordinator with request: '{request}' ---")
    user_id = "user_123"

    for attempt in range(1, max_retry + 2):
        session_id = str(uuid.uuid4())
        runner.session_service.create_session_sync(
            app_name=runner.app_name, user_id=user_id, session_id=session_id
        )
        try:
            final = None
            seen = 0
            for event in runner.run(
                user_id=user_id,
                session_id=session_id,
                new_message=types.Content(role="user", parts=[types.Part(text=request)]),
            ):
                seen += 1
                if verbose:
                    print(f"   [event {seen}] author={getattr(event, 'author', '?')} "
                          f"final={event.is_final_response()}")
                if event.is_final_response() and event.content:
                    final = extract_text(event.content)
                    break

            if final is None:
                print(f"   ⚠️ 收到 {seen} 个事件，但没有 final response；用 verbose=True 看事件流")
                return ""

            print(f"Coordinator Final Response: {final}")
            return final

        except Exception as e:
            msg = str(e)
            head = (msg.strip().splitlines() or [type(e).__name__])[0][:160]

            # 429 限流 和 503 高负载 都是【瞬时】的，都该重试。
            # 实测 gemini-3.1-flash-lite 的用例 A 连撞两次 503，而用例 B 一次就过。
            transient = ("429" in msg or "RESOURCE_EXHAUSTED" in msg
                         or "503" in msg or "UNAVAILABLE" in msg)
            if transient:
                if attempt <= max_retry:
                    wait = _retry_delay(msg)
                    print(f"   ⏳ 第 {attempt} 次撞瞬时错误，等 {wait}s 后重试（新 session）")
                    time.sleep(wait)
                    continue
                print(f"   ❌ 瞬时错误未恢复：{head}")
                return f"[TRANSIENT ERROR] {head}"

            print(f"   ❌ 出错({type(e).__name__}): {head}")
            return f"[ERROR] {head}"

    return ""


runner = InMemoryRunner(coordinator)

# 【适配 3】书里跑 4 个用例，免费层 5 RPM 撑不住；默认跑 2 个并中间 sleep 25s。
# 跑起来会看到几条 stderr（context_cache_config 提示、sync 接口弃用、"Task exception
# was never retrieved"），都不影响结果 —— 看 stdout 有没有 ">>> Booking Handler Called"。
DEMO_GAP_S = 0 if adk_backend() == "relay" else 25   # 中转站没有每分钟限流，不用等

DEMO = [
    ("A 预订", "Book me a hotel in Paris."),
    ("B 信息", "What is the highest mountain in the world?"),
    # ("C 信息", "Tell me a random fact."),
    # ("D 预订", "Find flights to Tokyo next month."),
]

VERBOSE = False        # ← 想看事件流就把这里改成 True

for i, (label, req) in enumerate(DEMO):
    if i > 0 and DEMO_GAP_S:
        print(f"\n(等待 {DEMO_GAP_S}s 避开免费层限流...)")
        time.sleep(DEMO_GAP_S)
    print(f"\n=== 用例 {label} ===")
    print("结果:", run_coordinator(runner, req, verbose=VERBOSE))

App "InMemoryRunner" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


Deprecated. Please migrate to the async method.


使用模型: backend=relay model=gemini-3.5-flash base_url=https://open.xiaojingai.com

=== 用例 A 预订 ===

--- Running Coordinator with request: 'Book me a hotel in Paris.' ---


  >>> Booking Handler Called


Deprecated. Please migrate to the async method.


Task exception was never retrieved
future: <Task finished name='Task-50' coro=<AsyncClient.aclose() done, defined at /home/stvli/agentic_adk_venv/lib/python3.12/site-packages/google/genai/client.py:233> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/home/stvli/agentic_adk_venv/lib/python3.12/site-packages/google/genai/client.py", line 258, in aclose
    await self._api_client.aclose()
  File "/home/stvli/agentic_adk_venv/lib/python3.12/site-packages/google/genai/_api_client.py", line 2819, in aclose
    await self._async_httpx_client.aclose()
  File "/home/stvli/agentic_adk_venv/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/home/stvli/agentic_adk_venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/home/stvli/agentic_adk_venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in ac

Coordinator Final Response: I have initiated your booking request for a hotel in Paris. Let me know if you need to provide any specific details like dates, budget, or preferences!
结果: I have initiated your booking request for a hotel in Paris. Let me know if you need to provide any specific details like dates, budget, or preferences!

=== 用例 B 信息 ===

--- Running Coordinator with request: 'What is the highest mountain in the world?' ---


  >>> Info Handler Called


Coordinator Final Response: The highest mountain in the world above sea level is **Mount Everest**. It is located in the Himalayas on the border between Nepal and China (Tibet). Its official elevation is 8,848.86 meters (29,031.7 feet). 

If you measure from base to peak, however, **Mauna Kea** in Hawaii is the tallest mountain, measuring over 10,210 meters (33,500 feet) from its underwater base to its peak.
结果: The highest mountain in the world above sea level is **Mount Everest**. It is located in the Himalayas on the border between Nepal and China (Tibet). Its official elevation is 8,848.86 meters (29,031.7 feet). 

If you measure from base to peak, however, **Mauna Kea** in Hawaii is the tallest mountain, measuring over 10,210 meters (33,500 feet) from its underwater base to its peak.
